# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ushah3984-web/Fly_rank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, subprocess

REPO_URL = "https://github.com/ushah3984-web/Fly_rank"
REPO_DIR = "Fly_rank"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("Now in:", os.getcwd())

Now in: /content/flyrank-ml-internship/Fly_rank


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!pip install datasets -q
from datasets import load_dataset
import pandas as pd

dataset_march = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    token=hf_token,
    data_files="fact_content_daily_performance/month=2026-03/*.parquet"
)
df_march = dataset_march["train"].to_pandas()
print(df_march.shape)

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

(9841378, 30)


In [6]:
dataset_content = load_dataset("FlyRank/internship-warehouse", "dim_content", token=hf_token)
df_content = dataset_content["train"].to_pandas()

df = df_march.merge(df_content[["content_hash_id", "content_type"]], on="content_hash_id", how="left")
df = df[df["gsc_data_available"] == True].copy()
df = df[df["gsc_impressions"] > 0].copy()
df["ctr"] = df["gsc_clicks"] / df["gsc_impressions"]

def position_tier(pos):
    if pos <= 3:
        return "top_3"
    elif pos <= 10:
        return "page_1"
    elif pos <= 20:
        return "page_3_5"
    else:
        return "deep"

df["position_tier"] = df["gsc_avg_position"].apply(position_tier)
expected_ctr = df.groupby("position_tier")["ctr"].transform("mean")
df["ctr_gap"] = df["ctr"] - expected_ctr
df["low_ctr_flag"] = (df["ctr_gap"] < 0).astype(int)

print(df.shape)
print(df["low_ctr_flag"].value_counts())

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/519606 [00:00<?, ? examples/s]

(3611061, 35)
low_ctr_flag
1    3257979
0     353082
Name: count, dtype: int64


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [7]:
from sklearn.model_selection import train_test_split

feature_cols = ["gsc_impressions", "gsc_clicks", "gsc_avg_position"]
X = df[feature_cols]
y = df["low_ctr_flag"]

# Group-aware split: keep all rows from the same client together in either train or test,
# so the model isn't tested on clients it has already partially seen.
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_hash_id"]))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train size:", X_train.shape[0], "Test size:", X_test.shape[0])


Train size: 2690999 Test size: 920062


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [8]:

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, roc_auc_score

# --- Model 1: Logistic Regression ---
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)
log_preds = log_model.predict_proba(X_test)[:, 1]

# --- Model 2: Random Forest ---
rf_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict_proba(X_test)[:, 1]

# --- Baseline: a genuinely SIMPLE, independent rule ---
# Uses only position (not ctr_gap) — a naive "better position = better CTR assumed" rule,
# so it's NOT circularly derived from the same value that built our target.
test_df = df.iloc[test_idx].copy()
test_df["baseline_score"] = -test_df["gsc_avg_position"]  # lower position number = higher score

def precision_at_k(scores, y_true, k=50):
    top_k_idx = pd.Series(scores).sort_values(ascending=False).head(k).index
    return y_true.iloc[top_k_idx].mean()

y_test_reset = y_test.reset_index(drop=True)
baseline_p50 = precision_at_k(test_df["baseline_score"].reset_index(drop=True), y_test_reset, k=50)
log_p50 = precision_at_k(log_preds, y_test_reset, k=50)
rf_p50 = precision_at_k(rf_preds, y_test_reset, k=50)

comparison = pd.DataFrame({
    "Method": ["Baseline rule (position only)", "Logistic Regression", "Random Forest"],
    "ROC AUC": [
        roc_auc_score(y_test, test_df["baseline_score"]),
        roc_auc_score(y_test, log_preds),
        roc_auc_score(y_test, rf_preds)
    ],
    "Precision@50": [baseline_p50, log_p50, rf_p50]
})
print(comparison)

                          Method   ROC AUC  Precision@50
0  Baseline rule (position only)  0.328903          0.98
1            Logistic Regression  0.996809          1.00
2                  Random Forest  0.999995          1.00


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Feature importances:\n", importances)

# Look at cases where the model was most confident but wrong
test_df["rf_pred"] = rf_preds
test_df["actual"] = y_test.values
errors = test_df[(test_df["rf_pred"] > 0.8) & (test_df["actual"] == 0)]
print("\nHigh-confidence wrong predictions:", errors.shape[0])
print(errors[["content_hash_id", "gsc_impressions", "gsc_avg_position", "ctr_gap", "rf_pred"]].head(10))

Feature importances:
 gsc_clicks          0.840731
gsc_impressions     0.137525
gsc_avg_position    0.021744
dtype: float64

High-confidence wrong predictions: 0
Empty DataFrame
Columns: [content_hash_id, gsc_impressions, gsc_avg_position, ctr_gap, rf_pred]
Index: []


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.